# AMIPS FIQA quickstart

This notebook runs the full AMIPS pipeline on [`BeIR/fiqa-generated-queries`](https://huggingface.co/datasets/BeIR/fiqa-generated-queries), a relatively small dataset (~56k keys)

Install the package first with `bash scripts/setup.sh`, launch Jupyter from the repository root, and run the cells top to bottom.

## Setup

Imports and shared paths. Data preparation (embedding and clustering) is run through the existing scripts; training and evaluation are inlined below.

In [ ]:
import pathlib

import jax
import jax.numpy as jnp
import numpy as np
import optax
import tqdm
from flax import nnx

from amips import config, data, metrics, models, training, utils

dataset = "BeIR/fiqa-generated-queries"
embeddings_dir = "/tmp/amips/fiqa/embeddings"
searcher_targets = "/tmp/amips/fiqa/targets_searcher"
router_targets = "/tmp/amips/fiqa/targets_router"

mesh = jax.make_mesh((jax.device_count(),), "data")
param_sharding = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec())
data_sharding = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec("data"))

### Training loop

`train` reproduces the loop in [`scripts/train_amips.py`](scripts/train_amips.py): it builds the model and optimizer, minimizes the AMIPS regression objective while keeping an EMA of the weights, and periodically evaluates on the held-out split. It returns the (EMA) model used for evaluation, the key metadata, and the resolved config. The same function trains the searcher and the router.

In [ ]:
def train(input_path: str, config_path: str):
    cfg = config.load_config(config_path)

    train_dl, keys_meta = data.build_dataloader(
        input_path,
        split="train",
        batch_size=cfg.dataloader.trn_batch_size,
        num_epochs=None,
        shuffle=True,
        drop_remainder=True,
        seed=cfg.seed,
        return_key_metadata=True,
    )
    test_dl, _ = data.build_dataloader(
        input_path,
        split="test",
        batch_size=cfg.dataloader.val_batch_size,
        num_epochs=1,
        shuffle=False,
        drop_remainder=False,
        seed=cfg.seed,
    )

    with jax.set_mesh(mesh):
        model = models.create_model(
            cfg,
            num_keys=keys_meta.num_keys,
            dim=keys_meta.dim,
            num_cls=keys_meta.num_cls,
            rngs=nnx.Rngs(cfg.seed + 1),
        )
        model = jax.device_put(model, param_sharding)
        num_params = models.count_params(model)
        print(
            f"params: {num_params} ({num_params / keys_meta.numel:.5f} of database size)"
        )

        lr_schedule = optax.warmup_cosine_decay_schedule(
            init_value=cfg.optimizer.init_value,
            peak_value=cfg.optimizer.peak_value,
            end_value=cfg.optimizer.end_value,
            warmup_steps=cfg.optimizer.warmup_steps,
            decay_steps=cfg.dataloader.num_trn_steps,
        )
        optimizer = nnx.Optimizer(
            model,
            optax.chain(
                optax.zero_nans(),
                optax.clip_by_global_norm(cfg.optimizer.grad_clip),
                optax.adam(lr_schedule),
            ),
            wrt=nnx.Param,
        )

        ema = (
            models.EMA(model, decay=cfg.model.ema_decay)
            if cfg.model.ema_decay is not None
            else None
        )
        eval_model = ema.model if ema is not None else model
        ema_sharding = param_sharding if ema is not None else None

        train_step_fn = nnx.jit(
            training.train_step,
            in_shardings=(param_sharding, param_sharding, ema_sharding, data_sharding),
            static_argnames=("score_weight", "target_weight"),
        )
        eval_step_fn = nnx.jit(
            training.eval_step,
            in_shardings=(param_sharding, data_sharding, param_sharding),
        )
        all_keys = jax.device_put(keys_meta.keys[:], param_sharding)

        for step, batch in enumerate(
            tqdm.tqdm(train_dl, desc="Training", total=cfg.dataloader.num_trn_steps)
        ):
            if step >= cfg.dataloader.num_trn_steps:
                break
            batch = jax.device_put(batch, data_sharding)
            train_step_fn(
                model,
                optimizer,
                ema,
                batch,
                cfg.loss.score_weight,
                cfg.loss.target_weight,
            )
            if (
                cfg.dataloader.eval_every
                and (step + 1) % cfg.dataloader.eval_every == 0
            ):
                eval_metrics = training.evaluate(
                    eval_model,
                    test_dl,
                    eval_step_fn=eval_step_fn,
                    all_keys=all_keys,
                    num_steps=cfg.dataloader.num_val_steps,
                )
                print(
                    "eval", step + 1, {k: round(v, 4) for k, v in eval_metrics.items()}
                )

    return eval_model, keys_meta, cfg

## Embed queries and keys

Encode the query and key text columns into L2-normalized vectors, shared by both models below and written to `embeddings_dir` as `queries.npy` and `keys.npy`.

In [ ]:
!python scripts/embed_data.py \
    --dataset {dataset} \
    --query_text_column query \
    --key_text_column text \
    --encoder all-MiniLM-L6-v2 \
    --output_path {embeddings_dir} \
    --batch_size 256

## Searcher

With `--num_clusters 1` there is a single partition, so this trains a pure searcher: a KeyNet that maps a query to the region of the key that maximizes the inner product.

In [ ]:
!python scripts/precompute_top1.py \
    --input_path {embeddings_dir} \
    --output_path {searcher_targets} \
    --num_clusters 1 \
    --num_noise_samples 50 \
    --noise_std 0.02 \
    --test_size 2048 \
    --batch_size 1024

### Train

In [ ]:
searcher_model, searcher_meta, searcher_cfg = train(
    input_path=searcher_targets,
    config_path="configs/train_amips_fiqa.yaml",
)

### Evaluate

Build a FAISS IVF index over the keys and sweep the number of probed cells, comparing recall for the raw query against the KeyNet-mapped query. Because the mapped query lands near the true key, it reaches high recall while scanning a small fraction of the database.

In [ ]:
searcher_model.eval()
k_values = (1, 10, 100)

test_dl, _ = data.build_dataloader(
    searcher_targets,
    split="test",
    batch_size=searcher_cfg.dataloader.val_batch_size,
    num_epochs=1,
    shuffle=False,
    drop_remainder=False,
    seed=searcher_cfg.seed,
)
natural, gt = [], []
for batch in test_dl:
    natural.append(np.asarray(batch["query"]))
    gt.append(np.asarray(batch["topk_indices"]).squeeze(-1))
natural = np.concatenate(natural)
gt = jnp.asarray(np.concatenate(gt))

with jax.set_mesh(mesh):
    projected = np.asarray(
        searcher_model.gradient(jax.device_put(natural, data_sharding))
    )

n_clusters = int(searcher_meta.num_keys**0.5)
index = utils.create_ivf_index(searcher_meta.keys[:], n_clusters=n_clusters)
nprobes = sorted({int(v) for v in np.geomspace(1, n_clusters, num=12)})

header = ["frac"] + [f"nat@{k}" for k in k_values] + [f"proj@{k}" for k in k_values]
print("".join(f"{h:>10}" for h in header))
for nprobe in nprobes:
    _, nat_idx = utils.search_index(index, natural, k=max(k_values), nprobe=nprobe)
    _, proj_idx = utils.search_index(index, projected, k=max(k_values), nprobe=nprobe)
    nat_idx, proj_idx = jnp.asarray(nat_idx), jnp.asarray(proj_idx)
    nat = [
        float(metrics.recall_at_k(nat_idx, gt, start=0, end=k).mean()) for k in k_values
    ]
    proj = [
        float(metrics.recall_at_k(proj_idx, gt, start=0, end=k).mean())
        for k in k_values
    ]
    print("".join(f"{v:>10.3f}" for v in [nprobe / n_clusters] + nat + proj))

## Router

With `--num_clusters > 1` the keys are partitioned and the model learns to route a query to the partitions that contain its answer. Here the KeyNet has one output head per partition, and its scores rank the partitions to probe.

In [ ]:
!python scripts/precompute_top1.py \
    --input_path {embeddings_dir} \
    --output_path {router_targets} \
    --num_clusters 10 \
    --num_noise_samples 50 \
    --noise_std 0.02 \
    --test_size 2048 \
    --batch_size 1024 \
    --max_iterations 300

### Train

In [ ]:
router_model, router_meta, router_cfg = train(
    input_path=router_targets,
    config_path="configs/train_amips_fiqa_router.yaml",
)

### Evaluate

Compare the learned router against the centroid baseline: for k = 1..num_clusters, the routing recall (true partition among the top-k probed) and the fraction of keys scanned. The learned router reaches a given recall while scanning fewer keys.

In [ ]:
router_model.eval()
n_cls = router_meta.num_cls

test_dl, _ = data.build_dataloader(
    router_targets,
    split="test",
    batch_size=router_cfg.dataloader.val_batch_size,
    num_epochs=1,
    shuffle=False,
    drop_remainder=False,
    seed=router_cfg.seed,
)
queries, gt_scores = [], []
for batch in test_dl:
    queries.append(jax.device_get(batch["query"]))
    gt_scores.append(jax.device_get(batch["score"]))
queries = np.concatenate(queries)
true_cluster = np.concatenate(gt_scores).argmax(1)

assignments = np.load(pathlib.Path(router_targets) / "cluster_assignments.npy")
centroids = np.load(pathlib.Path(router_targets) / "centroids.npy")
sizes = np.bincount(assignments, minlength=n_cls).astype(np.float64)

with jax.set_mesh(mesh):
    learned = np.asarray(router_model(jax.device_put(queries, data_sharding)))
qn = queries / np.linalg.norm(queries, axis=1, keepdims=True)
cn = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
centroid = qn @ cn.T


def recall_and_frac(scores: np.ndarray):
    order = np.argsort(-scores, axis=1)
    ranks = (order == true_cluster[:, None]).argmax(1) + 1
    recall = np.array([(ranks <= k).mean() for k in range(1, n_cls + 1)])
    frac = np.cumsum(sizes[order], axis=1).mean(0) / router_meta.num_keys
    return recall, frac


learned_recall, learned_frac = recall_and_frac(learned)
centroid_recall, centroid_frac = recall_and_frac(centroid)
header = ["clusters", "learned_frac", "learned_rec", "centroid_frac", "centroid_rec"]
print("".join(f"{h:>15}" for h in header))
for k in range(n_cls):
    row = [
        k + 1,
        learned_frac[k],
        learned_recall[k],
        centroid_frac[k],
        centroid_recall[k],
    ]
    print(f"{row[0]:>15}" + "".join(f"{v:>15.3f}" for v in row[1:]))